# Numerische Lösung des starren Pendels

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from IPython.core.pylabtools import figsize, getfigs
from IPython.display import Video, display
from numpy import *

# Verzeichnis für exportierte Abbildungen und Videos
notebook_name = "PhaseSpacePendulum"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "01_mechanik" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)
animation_display_width = 360
print(f"Output directory: {output_dir}")

## Hamiltonfunktion und Bewegungsgleichungen 


Die Hamiltonfunktion des starren Pendels ist 

$$H(p_\theta,\theta) =  \frac{p_\theta^2}{2m\ell^2} - mg\ell \cos\theta$$

Die Hamiltonschen Bewegungsgleichungen sind daher

$$ \dot \theta = \frac{\partial H}{\partial p_\theta} = \frac{ p_\theta }{m\ell^2},
\,\,\,\,\,
\dot p_\theta = -\frac{\partial H}{\partial \theta} = -mg\ell \sin\theta$$

Ineinander eingesetzt ergibt das die bekannte Gleichung zweiter Ordnung

$$\ddot \theta = - \Omega^2 \sin\theta,\,\,\,\,\text{mit}\,\,\,\Omega=\sqrt{\frac{g}{\ell}}. $$

### Dimensionslose Darstellung:

Für die folgende Rechung wählen wir "natürliche Einheiten" für Energie, Impuls, und Zeit:

Einheit Energie:  $[E] = mg\ell \equiv E_0$

Einheit Impuls:  $[p_\theta] = m\ell^2\Omega \equiv p_0$  

Einheit Zeit:  $[t] =1/\Omega \equiv t_0$

(Beachte: der zum Winkel $\theta$ konjugierte Impuls $p_\theta$ ist ein Drehimpuls mit der Einheit Länge $\times$ Masse $\times$ Geschwindigkeit)

Wir drücken alle Grössen durch entsprechende Einheiten aus: $E=\bar E \times E_0$ (Energie), $p_\theta = \bar p_\theta\times p_0$, $t = \tau \times t_0$, mit $\bar H$, $\bar p_0$, $\tau$, dimensionslos. De Gleichungen für die Dimensionslosen Grössen lauten dann (bitte nachrechnen): $ \frac{d}{d\tau} \theta =  \bar p_\theta$, 
$\frac{d}{d\tau} \bar p_\theta = -\sin\theta $ und die Energie ist gegeben durch $ \bar E = \frac{\bar p_\theta^2}{2} - \cos\theta$.

Das Arbeiten mit den dimensionslosen Grössen ist also analog dazu, dass wir in den Gleichungen oben $m=g=\ell=1$ setzen:


$$H =  \frac{p_\theta^2}{2} - \cos\theta,
\,\,\,\,
\dot \theta = p_\theta ,
\,\,\,\,\,
\dot p_\theta = -\sin\theta$$

und davon ausgehen, dass die Energie in Einheiten von $E_0$, Impuls in Einheiten von $p_0$ und Zeit in Einheiten von $t_0$ gemessen wird. 









In [ ]:
def pendulum_energy(theta, p):
    return 0.5* p**2 - np.cos(theta)

In [ ]:
# contour plot der Eneregie im Phasenraum


# der folgende code erzeugt einen Contourplot für die energie 
# (Kurven Energie = konstant) im Phasenraum

th_grid, p_grid= np.meshgrid(np.linspace(-4, 4, 256), np.linspace(-3, 3, 256))
H_grid = pendulum_energy(th_grid, p_grid)
# Energiewerte, für die Konturlinien gezeigt werden:
H_levels = np.linspace(-1,4,21)

# plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.contour(th_grid, p_grid, H_grid, levels=H_levels,colors='#BBBBBB',linestyles='solid')
plt.show()


In [ ]:
# Numerische Lösungsverfahren: Euler, Verlet:

def pendulum_euler_step(th_n,p_n,h):
    th_n1 = th_n + h*p_n
    p_n1 = p_n - h*np.sin(th_n)
    return th_n1, p_n1

def pendulum_euler(th_init,p_init,h,nmax):
    #timestep
    # arrays to store output
    th_t=np.zeros(nmax)
    p_t=np.zeros(nmax)
    th_t[0]=th_init
    p_t[0]=p_init
    # variable to store theta(t) and p(t) ar current time 
    thn=th_init
    pn=p_init
    for i in np.arange(1, nmax):
        thn,pn = pendulum_euler_step(thn,pn,h)
        th_t[i]=thn
        p_t[i]=pn
    return th_t,p_t


def pendulum_verlet_step(thn,pn12,pn,h):
    thn_1 = thn + h*pn12
    f=-np.sin(thn_1)
    pn12_1 = pn12 + h*f
    pn_1 = pn12 + 0.5*h*f
    return thn_1,pn12_1,pn_1

def pendulum_verlet(th_init,p_init,h,nmax):
    #timestep
    # arrays to store output
    th_t=np.zeros(nmax)
    p_t=np.zeros(nmax)
    th_t[0]=th_init
    p_t[0]=p_init
    #startup
    thn=th_init
    pn=p_init
    pn12=pn+0.5*h*(-np.sin(thn))
    for i in np.arange(1, nmax):
        thn,pn12,pn = pendulum_verlet_step(thn,pn12,pn,h)
        th_t[i]=thn
        p_t[i]=pn
    return th_t,p_t


# energie als Funktion von theta und p



In [ ]:
from matplotlib.animation import FuncAnimation


# Time series for (th_1, pth_1) generated by your program
nmax=400
#h=2*np.pi/100.0
h=0.1
# choose here Euler or verlet ...
q0=np.pi-0.01
th_1,pth_1=pendulum_verlet(q0,0,h,nmax) 
#th_1,pth_1=pendulum_euler(q0,0,h,nmax) 

# Example function (replace with your actual pendulum energy function)
def pendulum_energy(theta, p):
    return 0.5 * p**2 + (1 - np.cos(theta))

# Generate the grid for the contour plot
th_grid, p_grid = np.meshgrid(np.linspace(-4, 4, 256), np.linspace(-3, 3, 256))
H_grid = pendulum_energy(th_grid, p_grid)

# Energy levels for the contour plot
H_levels = np.linspace(-1, 4, 21)


# Create the contour plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.set_xlim(-4, 4)
ax.set_ylim(-3, 3)

# Draw the contour lines
contours = ax.contour(th_grid, p_grid, H_grid, levels=H_levels, colors='#BBBBBB', linestyles='solid')

# Initialize the curve to be drawn incrementally and the dot (current point)
line, = ax.plot([], [], color='red', lw=2)
dot, = ax.plot([], [], 'bo')  # 'bo' creates a blue dot

# Initialize function for the animation
def init():
    line.set_data([], [])
    dot.set_data([], [])
    return line, dot,

# Update function for the animation
def update(frame):
    # Draw the line from the first point up to the current frame
    line.set_data(th_1[:frame], pth_1[:frame])
    
    # Update the position of the dot to the current point
    dot.set_data([th_1[frame]], [pth_1[frame]])
    
    return line, dot,

# Create the animation
ani = FuncAnimation(fig, update, frames=len(th_1), init_func=init, interval=50, blit=True)

# Export and display the animation in the notebook
video_file = output_dir / 'pendulum_animation_with_dot.mp4'
ani.save(str(video_file), writer='ffmpeg', fps=20)
plt.close(fig)
display(Video(
    str(video_file), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))


In [ ]:
#Plot of the final trajectory with points:

# Create the contour plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-3, 3)

# Draw the contour lines
contours = ax.contour(th_grid, p_grid, H_grid, levels=H_levels, colors='#BBBBBB', linestyles='solid')
ax.plot(th_1,pth_1, '.', label=f't={0*h:.1f}')

plt.show()


In [ ]:

# Calculate the coordinates (x, y) for the moving circle
x_circle = np.sin(th_1)
y_circle = -np.cos(th_1)

# Create the plot
fig, ax = plt.subplots()
ax.set_xlim(-1.5, 1.5)  # Adjust the limits as needed
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')  # Keep the aspect ratio square

# Hide the axes
ax.axis('off')  # This turns off both the x and y axes

# Initialize the circle (marker) and the line to the origin
circle, = ax.plot([], [], 'bo', markersize=10)  # Blue circle
line, = ax.plot([], [], 'k-', lw=2)  # Black line from origin to circle

# Draw a static dashed circle with radius 1 around the origin
static_circle = plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='dashed')
ax.add_patch(static_circle)

# Initialize function for the animation
def init():
    circle.set_data([], [])
    line.set_data([], [])
    return circle, line,

# Update function for the animation
def update(frame):
    # Update the circle's position
    circle.set_data([x_circle[frame]], [y_circle[frame]])
    
    # Update the line from the origin to the circle's position
    line.set_data([0, x_circle[frame]], [0, y_circle[frame]])
    
    return circle, line,

# Create the animation
ani = FuncAnimation(fig, update, frames=len(th_1), init_func=init, interval=50, blit=True)

# Export and display the animation in the notebook
video_file = output_dir / 'circle_animation_no_axes.mp4'
ani.save(str(video_file), writer='ffmpeg', fps=20)
plt.close(fig)
display(Video(
    str(video_file), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))


## illustration phasenraumvolumen

In [ ]:
# define a curve of starting values in phase space 
# here: a circle of radius R around (th0,p0), with nmax points
def get_start_values(th0,p0,R,nstart):
    th_start=np.zeros(nstart)
    p_start=np.zeros(nstart)
    ds=2*np.pi/(nstart-1)
    for i in range(nstart):
        s=i*ds
        th_start[i]=th0+R*np.cos(s)
        p_start[i]=p0+R*np.sin(s)
    return th_start,p_start

# propagate a start value over nmax timesteps ... generates a gig file 

nmax=400
h=2*np.pi/100.0


nstart=100 
thetas=np.full((nmax, nstart), 0.0)    
ps=np.full((nmax, nstart), 0.0)    
th0=0
p0=1.6
R=0.2
thetas[0],ps[0]=get_start_values(th0,p0,R,nstart)

for j in range(nstart):
    #verlet startup
    thn=thetas[0][j]
    pn=ps[0][j]
    th_1,pth_1=pendulum_verlet(thn,pn,h,nmax) 
    for n in range(1,nmax):
        thetas[n][j]=th_1[n]
        ps[n][j]=pth_1[n]

        
# restore contour plot from above
# plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.set_xlim(-4, 4)
ax.set_ylim(-3, 3)
ax.contour(th_grid, p_grid, H_grid, levels=H_levels,colors='#BBBBBB',linestyles='solid')

#plot p(t) vs theta(t) for two periods:

for j0 in range(0,100,10):
    ax.plot(thetas[j0,:],ps[j0,:], 'b-')

#plt.legend()
plt.show()



In [ ]:
# restore contour plot from above
# plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.set_xlim(-4, 4)
ax.set_ylim(-3, 3)
ax.contour(th_grid, p_grid, H_grid, levels=H_levels,colors='#BBBBBB',linestyles='solid')
ax.plot(thetas[0,:],ps[0,:], 'k--')

    
# Initialize the curve that will be updated
line, = ax.plot([], [], color='red', lw=2)

# Initialize function for the animation
def init():
    line.set_data([], [])
    return line,

# Update function for the animation
def update(frame):
    # Update the curve for the current frame n
    line.set_data(thetas[frame, :], ps[frame, :])
    return line,

# Create the animation
ani = FuncAnimation(fig, update, frames=nmax, init_func=init, interval=100, blit=True)

# Export and display the animation in the notebook
video_file = output_dir / 'curve_on_contour_animation.mp4'
ani.save(str(video_file), writer='ffmpeg', fps=20)
plt.close(fig)
display(Video(
    str(video_file), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))

# Illustration Diekrete Abbildung Phasenraumvolumen:

In [ ]:
# define a curve of starting values in phase space 
# here: a square of size 2R around (th0,p0), with 4*nstart points
def get_start_values(th0,p0,R,nstart):
    th_start=np.zeros(4*nstart)
    p_start=np.zeros(4*nstart)
    ds=2.0*R/nstart
    for i in range(nstart):
        th_start[0*nstart+i]=th0-R
        p_start[0*nstart+i]=p0-R+i*ds
        th_start[1*nstart+i]=th0-R+i*ds
        p_start[1*nstart+i]=p0+R
        th_start[2*nstart+i]=th0+R
        p_start[2*nstart+i]=p0+R-i*ds
        th_start[3*nstart+i]=th0+R-i*ds
        p_start[3*nstart+i]=p0-R        
    return th_start,p_start

# propagate a start value over nmax timesteps ... generates a gig file 

nmax=10
h=1

nstart=5 
thetas=np.full((nmax, 4*nstart), 0.0)    
ps=np.full((nmax, 4*nstart), 0.0)    
th0=1
p0=0
R=0.2
thetas[0],ps[0]=get_start_values(th0,p0,R,nstart)

for j in range(4*nstart):
    thn=thetas[0][j]
    pn=ps[0][j]
    th_1,pth_1=pendulum_verlet(thn,pn,h,nmax) 
    for n in range(1,nmax):
        thetas[n][j]=th_1[n]
        ps[n][j]=pth_1[n]

        
# restore contour plot from above
# plot
fig, ax = plt.subplots()
ax.set_xlabel("theta")
ax.set_ylabel("p")
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-3, 3)
ax.contour(th_grid, p_grid, H_grid, levels=H_levels,colors='#BBBBBB',linestyles='solid')

#plot p(t) vs theta(t) for two periods:

ax.plot(thetas[0,:],ps[0,:], '.', label=f't={0*h:.1f}')
ax.plot(thetas[1,:],ps[1,:], '.', label=f't={1*h:.1f}')
ax.plot(thetas[2,:],ps[2,:], '.', label=f't={2*h:.1f}')
ax.plot(thetas[3,:],ps[3,:], '.', label=f't={3*h:.1f}')
ax.plot(thetas[4,:],ps[4,:], '.', label=f't={4*h:.1f}')
ax.plot(thetas[5,:],ps[5,:], '.', label=f't={5*h:.1f}')
ax.plot(thetas[6,:],ps[6,:], '.', label=f't={6*h:.1f}')

plt.legend()
plt.show()


